# Tool XY Offset Alignment

This notebook calibrates the **XY tool offset** (the `G10 P{n} X{dx} Y{dy}` values in `toffsets.g`) for a new tool mounted on the Jubilee.

**Concept:** pick up a reference tool (tool 0 — typically the pen/probe whose offset is already known), jog its tip to a fixed reference mark on the bed, record the position. Then swap to the tool being calibrated and center its tip on the same mark. The difference is the offset.

**Before starting:**
- Home the machine.
- Make sure the reference mark is visible (a crosshair sticker, calibration target, or the bed centre).
- Make sure `tpre/tpost/tfree` files for both tools are already on the Duet.
- Clear any tall labware from the bed.

## 1. Connect to the machine

In [ ]:
from science_jubilee.machine_session import MachineSession

# Hardware: replace the IP with your Duet's address
# session = MachineSession.hardware("192.168.1.2")

# Mock (no hardware needed — runs the full notebook logic offline)
session = MachineSession.mock()

nav = session.free_navigator
tc  = session.tool_changer

print("Connected. Available tool slots:")
for idx, tool in tc.tools.items():
    print(f"  slot {idx}: {tool}")

## 2. Configuration

- `REF_TOOL_IDX` — slot of the reference tool whose offset is already trusted (usually 0).
- `NEW_TOOL_IDX` — slot of the tool you are calibrating.
- `REF_Z` — Z height (mm) at which centering is done. Low enough to see the tip, high enough not to crash. Typically 50–100 mm above the bed.
- `APPROACH_XY` — approximate XY position of your reference mark.

In [ ]:
REF_TOOL_IDX  = 0      # reference tool slot (offset already known/trusted)
NEW_TOOL_IDX  = 1      # tool slot being calibrated
REF_Z         = 80.0   # Z height for visual centering (mm)
APPROACH_XY   = (150.0, 150.0)  # approximate location of reference mark on bed

## 3. Step A — center the reference tool

Run the cell below to home, pick up the reference tool, and move to the approach position.
Then **jog using DWC** (or the CalibrationJoystick) until the reference tool tip is exactly centred over the reference mark.
Run the recording cell when done.

In [ ]:
nav.home_all()
nav.pickup_tool(REF_TOOL_IDX)
nav.move_to(x=APPROACH_XY[0], y=APPROACH_XY[1], z=REF_Z, speed=6000)
print(f"Reference tool (slot {REF_TOOL_IDX}) in position. Jog to centre over mark, then run the next cell.")

In [ ]:
pos = nav.get_position()
ref_x, ref_y = pos["X"], pos["Y"]
print(f"Reference position: X={ref_x:.3f}  Y={ref_y:.3f}")

## 4. Step B — center the new tool

The machine will park the reference tool and pick up the tool being calibrated.
Jog again until its tip is centred over the **same** reference mark, then record.

In [ ]:
nav.park_tool()
nav.pickup_tool(NEW_TOOL_IDX)
nav.move_to(x=APPROACH_XY[0], y=APPROACH_XY[1], z=REF_Z, speed=6000)
print(f"New tool (slot {NEW_TOOL_IDX}) in position. Jog to centre over the same mark, then run the next cell.")

In [ ]:
pos = nav.get_position()
new_x, new_y = pos["X"], pos["Y"]
print(f"New tool position:  X={new_x:.3f}  Y={new_y:.3f}")

## 5. Compute the XY offset

The offset is simply $\Delta X = X_{ref} - X_{new}$, $\Delta Y = Y_{ref} - Y_{new}$.

The reference tool's own offset is already baked into $X_{ref}$/$Y_{ref}$ via the Duet's tool-offset system, so this difference is the absolute offset for the new tool in the Duet frame.

In [ ]:
dx = ref_x - new_x
dy = ref_y - new_y

# Get the existing Z offset for the new tool so we don't overwrite it
existing = tc.get_tool_offset(NEW_TOOL_IDX)
dz = existing[2] if existing else 0.0

print(f"Computed XY offset for slot {NEW_TOOL_IDX}:")
print(f"  dX = {dx:+.3f} mm")
print(f"  dY = {dy:+.3f} mm")
print()
print("Add this line to toffsets.g on the Duet (keep your existing Z value):")
print(f"  G10 P{NEW_TOOL_IDX} X{dx:.3f} Y{dy:.3f} Z{dz:.3f}  ; {tc.tools[NEW_TOOL_IDX].name}")

## 6. Apply temporarily and validate

Send the offset to the Duet now (takes effect immediately without reboot).
Then do a quick validation pass: pick up the new tool, move to the reference mark, and check visually that the tip lands on centre.

In [ ]:
gcode = f"G10 P{NEW_TOOL_IDX} X{dx:.3f} Y{dy:.3f} Z{dz:.3f}"
session.transport.send_gcode(gcode)
print(f"Sent: {gcode}")

nav.park_tool()
nav.pickup_tool(NEW_TOOL_IDX)
nav.move_to(x=APPROACH_XY[0], y=APPROACH_XY[1], z=REF_Z, speed=6000)
print("New tool is at the reference position. Verify the tip lands on the mark.")

In [ ]:
# Write the permanent G10 line to toffsets.g and upload to the Duet
# Uncomment when you are happy with the result.

# from science_jubilee._paths import jubilee_dir
# toffsets_path = jubilee_dir() / "firmware" / "sys" / "toffsets.g"
#
# new_line = f"G10 P{NEW_TOOL_IDX} X{dx:.3f} Y{dy:.3f} Z{dz:.3f}  ; {tc.tools[NEW_TOOL_IDX].name}\n"
# lines = toffsets_path.read_text().splitlines(keepends=True)
# pattern = f"G10 P{NEW_TOOL_IDX} "
# lines = [new_line if l.startswith(pattern) else l for l in lines]
# if new_line not in lines:
#     lines.append(new_line)
# toffsets_path.write_text("".join(lines))
# print(f"Updated {toffsets_path}")
#
# session.transport.upload_file(str(toffsets_path), destination="sys", remote_name="toffsets.g")
# print("Uploaded toffsets.g to Duet.")

print("Uncomment the block above to save and upload toffsets.g automatically.")
nav.park_tool()
print("Done — tool parked.")